# 2.7 — Advanced Retrieval

Basic retrieval (`similarity_search`) works well, but has weaknesses:
- Returns duplicate/redundant chunks
- Misses exact keyword matches (e.g. acronyms, names)
- No re-scoring after retrieval

This notebook covers three improvements:

| Technique | Problem it solves |
|-----------|------------------|
| **MMR** (Maximal Marginal Relevance) | Removes redundant chunks |
| **BM25** (Keyword Search) | Catches exact keyword matches |
| **Hybrid Search** (BM25 + Semantic) | Best of both worlds |
| **LLM Reranking** | Re-scores chunks for precision |

In [1]:
!pip install langchain langchain-ollama langchain-community chromadb rank_bm25 --quiet

In [2]:
!pip install -U langchain langchain-community langchain-core --quiet

## Setup — Documents & Vector Store

In [12]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

docs = [
    Document(page_content='All full-time employees receive 20 days of annual leave per year.', metadata={'section': 'Leave'}),
    Document(page_content='Annual leave entitlement is 20 days for permanent staff members.', metadata={'section': 'Leave'}),
    Document(page_content='Sick leave is up to 10 days per year with a medical certificate.', metadata={'section': 'Leave'}),
    Document(page_content='Parental leave is 16 weeks fully paid for primary caregivers.', metadata={'section': 'Leave'}),
    Document(page_content='Leave requests must be submitted at least 2 weeks in advance.', metadata={'section': 'Leave'}),
    Document(page_content='Employees may work remotely up to 3 days per week.', metadata={'section': 'Remote'}),
    Document(page_content='Remote workers must be available during core hours: 10am to 3pm.', metadata={'section': 'Remote'}),
    Document(page_content='All remote work equipment is provided by the company.', metadata={'section': 'Remote'}),
    Document(page_content='Health insurance is provided for all full-time employees and their immediate family.', metadata={'section': 'Benefits'}),
    Document(page_content='A gym membership subsidy of $50 per month is available.', metadata={'section': 'Benefits'}),
    Document(page_content='Employees receive a $1,000 annual learning and development budget.', metadata={'section': 'Benefits'}),
    Document(page_content='Standard working hours are 9am to 5pm, Monday to Friday.', metadata={'section': 'Hours'}),
    Document(page_content='Overtime must be pre-approved and compensated at 1.5x the hourly rate.', metadata={'section': 'Hours'}),
]

embeddings = OllamaEmbeddings(model='llama3.1')
vectorstore = Chroma.from_documents(docs, embedding=embeddings)

print(f'Vector store ready with {len(docs)} documents')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store ready with 13 documents


## 1. Standard Similarity Search (Baseline)

The default retriever — finds the top-k most similar chunks. Problem: can return **redundant** results.

In [13]:
query = 'How many leave days do employees get?'

standard_results = vectorstore.similarity_search(query, k=5)

print(f'Query: "{query}"')
print('\nStandard Similarity Search results:')
for i, doc in enumerate(standard_results):
    print(f'  {i+1}. [{doc.metadata["section"]}] {doc.page_content}')

print('\nNotice: first two results are very similar (redundant)!')

Query: "How many leave days do employees get?"

Standard Similarity Search results:
  1. [Leave] All full-time employees receive 20 days of annual leave per year.
  2. [Leave] All full-time employees receive 20 days of annual leave per year.
  3. [Remote] Employees may work remotely up to 3 days per week.
  4. [Remote] Employees may work remotely up to 3 days per week.
  5. [Benefits] Employees receive a $1,000 annual learning and development budget.

Notice: first two results are very similar (redundant)!


## 2. MMR — Maximal Marginal Relevance

MMR balances **relevance** and **diversity**. It avoids returning near-duplicate chunks.

```
MMR score = λ × similarity(doc, query) - (1 - λ) × max_similarity(doc, already_selected)
```

- `lambda_mult=1.0` → pure similarity (same as standard)
- `lambda_mult=0.0` → pure diversity
- `lambda_mult=0.5` → balanced (recommended)

In [14]:
# MMR retriever — diversity-aware
mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,
        'fetch_k': 8,       # fetch more candidates, then re-rank for diversity
        'lambda_mult': 0.5  # 0=diversity, 1=relevance
    }
)

mmr_results = mmr_retriever.invoke(query)

print(f'Query: "{query}"')
print('\nMMR results (diverse):')
for i, doc in enumerate(mmr_results):
    print(f'  {i+1}. [{doc.metadata["section"]}] {doc.page_content}')

print('\nNotice: results now cover different sections — no redundancy!')

Query: "How many leave days do employees get?"

MMR results (diverse):
  1. [Leave] All full-time employees receive 20 days of annual leave per year.
  2. [Remote] Employees may work remotely up to 3 days per week.
  3. [Benefits] Employees receive a $1,000 annual learning and development budget.

Notice: results now cover different sections — no redundancy!


## 3. BM25 — Keyword Search

BM25 is a classical keyword-ranking algorithm. It scores documents by **term frequency** and **inverse document frequency**.

Great for:
- Exact word matches (names, IDs, acronyms)
- Queries where keywords matter more than meaning

In [15]:
from langchain_community.retrievers import BM25Retriever

# BM25 works on raw documents (no embedding needed)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3

# Test with exact keyword query
keyword_query = 'parental leave 16 weeks'

bm25_results = bm25_retriever.invoke(keyword_query)
print(f'BM25 Query: "{keyword_query}"')
print('Results:')
for i, doc in enumerate(bm25_results):
    print(f'  {i+1}. [{doc.metadata["section"]}] {doc.page_content}')

print()

# Compare: semantic search on the same query
semantic_results = vectorstore.similarity_search(keyword_query, k=3)
print(f'Semantic Query: "{keyword_query}"')
print('Results:')
for i, doc in enumerate(semantic_results):
    print(f'  {i+1}. [{doc.metadata["section"]}] {doc.page_content}')

BM25 Query: "parental leave 16 weeks"
Results:
  1. [Leave] Parental leave is 16 weeks fully paid for primary caregivers.
  2. [Leave] Leave requests must be submitted at least 2 weeks in advance.
  3. [Leave] Annual leave entitlement is 20 days for permanent staff members.

Semantic Query: "parental leave 16 weeks"
Results:
  1. [Leave] Parental leave is 16 weeks fully paid for primary caregivers.
  2. [Leave] Parental leave is 16 weeks fully paid for primary caregivers.
  3. [Leave] Sick leave is up to 10 days per year with a medical certificate.


## 4. Hybrid Search — BM25 + Semantic (Best of Both)

**EnsembleRetriever** combines multiple retrievers by weighted score fusion.

```
final_score = weight_bm25 × bm25_score + weight_semantic × semantic_score
```

This catches both:
- Exact keyword matches (BM25)
- Conceptually related content (semantic)

In [16]:
from langchain_classic.retrievers import EnsembleRetriever

semantic_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
bm25_retriever.k = 3

# Combine with equal weighting
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.5, 0.5]   # adjust to favour one method
)

# Test queries
test_queries = [
    'How many vacation days?',        # semantic wins
    'overtime 1.5x rate',             # BM25 wins (exact numbers)
    'gym subsidy $50',                # BM25 wins (exact value)
]

for q in test_queries:
    results = hybrid_retriever.invoke(q)
    print(f'Query: "{q}"')
    for doc in results[:2]:
        print(f'  → [{doc.metadata["section"]}] {doc.page_content[:70]}...')
    print()

Query: "How many vacation days?"
  → [Benefits] Employees receive a $1,000 annual learning and development budget....
  → [Hours] Overtime must be pre-approved and compensated at 1.5x the hourly rate....

Query: "overtime 1.5x rate"
  → [Hours] Overtime must be pre-approved and compensated at 1.5x the hourly rate....
  → [Hours] Standard working hours are 9am to 5pm, Monday to Friday....

Query: "gym subsidy $50"
  → [Benefits] A gym membership subsidy of $50 per month is available....
  → [Hours] Overtime must be pre-approved and compensated at 1.5x the hourly rate....



## 5. LLM Reranking

After retrieving candidates, ask the LLM to **score and rerank** them by relevance to the query.

This adds a second pass of intelligence — the LLM understands context better than pure vector similarity.

```
Query → Retrieve 6 candidates → LLM scores each → Return top 3
```

In [17]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model='llama3.1', temperature=0)

rerank_prompt = ChatPromptTemplate.from_template("""
You are a relevance scorer. Given a query and a document, output a relevance score from 0 to 10.
Output ONLY the number, nothing else.

Query: {query}
Document: {document}

Score (0-10):
""")

def rerank(query: str, candidates: list, top_k: int = 3) -> list:
    """Score each candidate with LLM and return top_k."""
    scored = []
    for doc in candidates:
        response = (rerank_prompt | llm | StrOutputParser()).invoke({
            'query': query,
            'document': doc.page_content
        })
        try:
            score = float(response.strip())
        except ValueError:
            score = 0.0
        scored.append((score, doc))
        print(f'  Score {score:.0f}/10 — {doc.page_content[:60]}...')

    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


query = 'What leave benefits does the company offer?'

# Step 1: Retrieve more candidates than needed
candidates = vectorstore.similarity_search(query, k=6)
print(f'Query: "{query}"')
print(f'\nScoring {len(candidates)} candidates...')
top_docs = rerank(query, candidates, top_k=3)

print('\nTop 3 after reranking:')
for i, doc in enumerate(top_docs):
    print(f'  {i+1}. [{doc.metadata["section"]}] {doc.page_content}')

Query: "What leave benefits does the company offer?"

Scoring 6 candidates...
  Score 2/10 — Employees may work remotely up to 3 days per week....
  Score 2/10 — Employees may work remotely up to 3 days per week....
  Score 4/10 — Employees receive a $1,000 annual learning and development b...
  Score 4/10 — Employees receive a $1,000 annual learning and development b...
  Score 0/10 — Overtime must be pre-approved and compensated at 1.5x the ho...
  Score 0/10 — Overtime must be pre-approved and compensated at 1.5x the ho...

Top 3 after reranking:
  1. [Benefits] Employees receive a $1,000 annual learning and development budget.
  2. [Benefits] Employees receive a $1,000 annual learning and development budget.
  3. [Remote] Employees may work remotely up to 3 days per week.


## 6. Putting It All Together — Advanced RAG Chain

In [18]:
from langchain_core.runnables import RunnablePassthrough

answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant. Answer using only the context below.
If not in context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:
""")

def advanced_rag(query: str) -> str:
    # Step 1: Hybrid retrieval (broad, diverse)
    candidates = hybrid_retriever.invoke(query)

    # Step 2: LLM reranking (precision)
    top_docs = rerank(query, candidates, top_k=3)

    # Step 3: Generate answer
    context = '\n\n'.join(doc.page_content for doc in top_docs)
    chain = answer_prompt | llm | StrOutputParser()
    return chain.invoke({'context': context, 'question': query})


questions = [
    'How many days of sick leave can I take?',
    'What is the overtime pay rate?',
]

for q in questions:
    print(f'\nQ: {q}')
    answer = advanced_rag(q)
    print(f'A: {answer}')


Q: How many days of sick leave can I take?
  Score 8/10 — Sick leave is up to 10 days per year with a medical certific...
  Score 5/10 — All full-time employees receive 20 days of annual leave per ...
  Score 0/10 — A gym membership subsidy of $50 per month is available....
  Score 2/10 — Annual leave entitlement is 20 days for permanent staff memb...
A: Up to 10 days per year with a medical certificate.

Q: What is the overtime pay rate?
  Score 8/10 — Overtime must be pre-approved and compensated at 1.5x the ho...
  Score 0/10 — A gym membership subsidy of $50 per month is available....
  Score 0/10 — All remote work equipment is provided by the company....
A: 1.5x the hourly rate.


## Summary

| Technique | When to use | Key parameter |
|-----------|-------------|---------------|
| **Standard** | Simple use cases | `k` |
| **MMR** | Avoid redundant chunks | `lambda_mult`, `fetch_k` |
| **BM25** | Exact keywords, IDs, numbers | `k` |
| **Hybrid** | General production use | `weights=[0.5, 0.5]` |
| **Reranking** | High-precision Q&A | `top_k` after wider fetch |

**Recommended production setup:**
```
Hybrid (BM25 + Semantic) → Retrieve top-8 → Rerank → Take top-3 → LLM
```